In [1]:
import os
import sys
import subprocess

REPO_URL = "https://github.com/shashvat-dubey/355m-LLm-Training"
REPO_DIR = "/kaggle/working/355m-LLm-Training"

# ------------------------------------------------------------
# Clone repository
# ------------------------------------------------------------

if not os.path.exists(REPO_DIR):

    print("Cloning repository...")

    subprocess.run(
        [
            "git",
            "clone",
            REPO_URL,
            REPO_DIR,
        ],
        check=True,
    )

else:

    print("Repository already exists.")
    
    subprocess.run(
        ["git", "-C", REPO_DIR, "pull"],
        check=True,
    )


# ------------------------------------------------------------
# Add repository to Python path
# ------------------------------------------------------------

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

os.chdir(REPO_DIR)

print()
print("Repository:", REPO_DIR)
print("Working directory:", os.getcwd())

Cloning repository...


Cloning into '/kaggle/working/355m-LLm-Training'...



Repository: /kaggle/working/355m-LLm-Training
Working directory: /kaggle/working/355m-LLm-Training


In [2]:
import os
import shutil
import subprocess

REPO_DIR = "/kaggle/working/355m-LLm-Training"
DOWNLOAD_DIR = "/kaggle/working/checkpoint"
CHECKPOINT_DIR = os.path.join(REPO_DIR, "checkpoints")
CHECKPOINT_PATH = os.path.join(CHECKPOINT_DIR, "checkpoint_latest.pt")

DATASET_ID = "susuzim/gpt355m-checkpoint-500"

# --------------------------------------------------
# Clean previous checkpoint download
# --------------------------------------------------

if os.path.exists(DOWNLOAD_DIR):
    print("Existing download directory found. Replacing it...")
    shutil.rmtree(DOWNLOAD_DIR)

os.makedirs(DOWNLOAD_DIR, exist_ok=True)

# --------------------------------------------------
# Download latest Kaggle Dataset version
# --------------------------------------------------

print(f"Downloading: {DATASET_ID}")

subprocess.run(
    [
        "kaggle",
        "datasets",
        "download",
        "-d",
        DATASET_ID,
        "-p",
        DOWNLOAD_DIR,
        "--unzip",
        "--force",
    ],
    check=True,
)

# --------------------------------------------------
# Locate checkpoint
# --------------------------------------------------

downloaded_checkpoint = os.path.join(
    DOWNLOAD_DIR,
    "checkpoint_latest.pt",
)

if not os.path.exists(downloaded_checkpoint):
    print("Files downloaded:")
    for root, dirs, files in os.walk(DOWNLOAD_DIR):
        for file in files:
            print(os.path.join(root, file))

    raise FileNotFoundError(
        "checkpoint_latest.pt was not found after Kaggle download."
    )

# --------------------------------------------------
# Replace local checkpoint
# --------------------------------------------------

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

if os.path.exists(CHECKPOINT_PATH):
    print("Existing local checkpoint found. Replacing it...")
    os.remove(CHECKPOINT_PATH)

# Move rather than copy -> avoids needing another 4 GB temporarily.
os.replace(
    downloaded_checkpoint,
    CHECKPOINT_PATH,
)

# Remove download directory and leftover zip/metadata.
shutil.rmtree(DOWNLOAD_DIR, ignore_errors=True)

# --------------------------------------------------
# Verify
# --------------------------------------------------

size_gb = os.path.getsize(CHECKPOINT_PATH) / (1024 ** 3)

print()
print("=" * 50)
print("CHECKPOINT READY")
print("=" * 50)
print("Path:", CHECKPOINT_PATH)
print(f"Size: {size_gb:.2f} GB")
print("Exists:", os.path.exists(CHECKPOINT_PATH))

Downloading: susuzim/gpt355m-checkpoint-500
Dataset URL: https://www.kaggle.com/datasets/susuzim/gpt355m-checkpoint-500
License(s): CC0-1.0


100%|██████████| 3.64G/3.64G [00:18<00:00, 213MB/s]




CHECKPOINT READY
Path: /kaggle/working/355m-LLm-Training/checkpoints/checkpoint_latest.pt
Size: 3.97 GB
Exists: True


In [3]:
import os
import torch

path = "/kaggle/working/355m-LLm-Training/checkpoints/checkpoint_latest.pt"

checkpoint = torch.load(
    path,
    map_location="cpu",
    weights_only=False,
)


print("Local checkpoint step:", checkpoint["step"])
print("Local checkpoint loss:", checkpoint["loss"])

Local checkpoint step: 7500
Local checkpoint loss: 3.765685498714447


In [4]:
%%writefile kaggle_config.py

# ============================================================
# Kaggle Training Configuration
# ============================================================

# Persistent Kaggle Dataset
DATASET_ID = "susuzim/gpt355m-checkpoint-500"

# ------------------------------------------------------------
# Batch duration
# ------------------------------------------------------------

# Number of optimizer steps to perform in THIS Kaggle session.
#
# TEST:
#     5
#
# PRODUCTION:
#     5000
#
STEPS_PER_BATCH = 15000

# ------------------------------------------------------------
# Checkpoint upload frequency
# ------------------------------------------------------------

# Upload checkpoint to Kaggle every N GLOBAL training steps.
#
# TEST:
#     5
#
# PRODUCTION:
#     500
#
UPLOAD_INTERVAL = 500

# ------------------------------------------------------------
# Local checkpoint behavior
# ------------------------------------------------------------

CHECKPOINT_DIR = "checkpoints"

# Only keep one local checkpoint:
#     checkpoints/checkpoint_latest.pt
MAX_LOCAL_CHECKPOINTS = 1

# ------------------------------------------------------------
# Model training configuration
# ------------------------------------------------------------

BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 16

PRECISION = "fp16"
USE_TF32 = False

LEARNING_RATE = 3e-4
WEIGHT_DECAY = 0.1

BETA1 = 0.9
BETA2 = 0.95
EPS = 1e-8

WARMUP_STEPS = 2000
SCHEDULER_MAX_STEPS = 100_000
MIN_LR_RATIO = 0.1

MAX_GRAD_NORM = 1.0

# ------------------------------------------------------------
# Data
# ------------------------------------------------------------

DATASET_NAME = "HuggingFaceFW/fineweb-edu"
DATASET_SUBSET = "sample-10BT"
STREAMING = True
NUM_WORKERS = 0

# ------------------------------------------------------------
# Logging
# ------------------------------------------------------------

LOG_INTERVAL = 10

Writing kaggle_config.py


In [5]:
%%writefile kaggle_trainer.py

import json
import os
import subprocess
import time

import torch
import torch.distributed as dist
from tqdm import tqdm

from training.trainer import Trainer


class KaggleTrainer(Trainer):

    def __init__(
        self,
        *args,
        upload_dataset_id,
        upload_interval,
        **kwargs,
    ):
        super().__init__(*args, **kwargs)

        self.upload_dataset_id = upload_dataset_id
        self.upload_interval = upload_interval

        self.checkpoint_directory = (
            self.checkpoint_manager.directory
        )

        self.latest_checkpoint_path = os.path.join(
            self.checkpoint_directory,
            "checkpoint_latest.pt",
        )

    # --------------------------------------------------------
    # Save one rolling local checkpoint
    # --------------------------------------------------------

    def save_latest_checkpoint(self, loss):

        if not self.is_main_process:
            return

        if self.checkpoint_manager is None:
            raise RuntimeError(
                "Checkpoint manager is required."
            )

        model_to_save = self.model

        if hasattr(model_to_save, "module"):
            model_to_save = model_to_save.module

        # CheckpointManager creates checkpoint_<step>.pt
        self.checkpoint_manager.save(
            step=self.global_step,
            model=model_to_save,
            optimizer=self.optimizer,
            scheduler=self.scheduler,
            scaler=self.scaler,
            loss=loss,
        )

        numbered_checkpoint = os.path.join(
            self.checkpoint_directory,
            f"checkpoint_{self.global_step}.pt",
        )

        if not os.path.exists(numbered_checkpoint):
            raise FileNotFoundError(
                f"Expected checkpoint was not created: "
                f"{numbered_checkpoint}"
            )

        # Atomic replacement of latest checkpoint.
        os.replace(
            numbered_checkpoint,
            self.latest_checkpoint_path,
        )

        # Remove any other numbered checkpoints.
        for filename in os.listdir(
            self.checkpoint_directory
        ):
            if (
                filename.startswith("checkpoint_")
                and filename.endswith(".pt")
                and filename != "checkpoint_latest.pt"
            ):
                try:
                    os.remove(
                        os.path.join(
                            self.checkpoint_directory,
                            filename,
                        )
                    )
                except FileNotFoundError:
                    pass

        print(
            f"\nCheckpoint updated: "
            f"step {self.global_step} | "
            f"loss {loss:.6f}"
        )

    # --------------------------------------------------------
    # Upload latest checkpoint to Kaggle Dataset
    # --------------------------------------------------------

    def upload_latest_checkpoint(self, loss):

        if not self.is_main_process:
            return

        if not os.path.exists(
            self.latest_checkpoint_path
        ):
            raise FileNotFoundError(
                "checkpoint_latest.pt does not exist."
            )

        # Kaggle Dataset metadata.
        metadata = {
            "title": "GPT-355M Training Checkpoint",
            "id": self.upload_dataset_id,
            "licenses": [
                {
                    "name": "CC0-1.0"
                }
            ],
            "description": (
                "Persistent rolling checkpoint for "
                "GPT-355M training. "
                f"Current training step: {self.global_step}. "
                f"Training loss: {loss:.6f}."
            ),
        }

        metadata_path = os.path.join(
            self.checkpoint_directory,
            "dataset-metadata.json",
        )

        with open(
            metadata_path,
            "w",
            encoding="utf-8",
        ) as file:
            json.dump(
                metadata,
                file,
                indent=2,
            )

        message = (
            f"GPT-355M | "
            f"Step {self.global_step} | "
            f"Loss {loss:.6f}"
        )

        print(
            f"\nUploading checkpoint to Kaggle..."
        )
        print(f"Dataset: {self.upload_dataset_id}")
        print(f"Message: {message}")

        subprocess.run(
            [
                "kaggle",
                "datasets",
                "version",
                "-p",
                self.checkpoint_directory,
                "-m",
                message,
            ],
            check=True,
        )

        print(
            f"Kaggle upload complete: "
            f"step {self.global_step}"
        )

    # --------------------------------------------------------
    # Save + upload with DDP synchronization
    # --------------------------------------------------------

    def checkpoint_and_upload(self, loss):

        success = True
        error_message = ""

        if self.is_main_process:
            try:
                self.save_latest_checkpoint(loss)
                self.upload_latest_checkpoint(loss)

            except Exception as exc:
                success = False
                error_message = str(exc)

        # Tell every process whether rank 0 succeeded.
        if (
            dist.is_available()
            and dist.is_initialized()
        ):

            status = torch.tensor(
                [1 if success else 0],
                dtype=torch.int32,
                device=self.device,
            )

            dist.broadcast(
                status,
                src=0,
            )

            if status.item() == 0:

                if self.is_main_process:
                    print(
                        "\nCheckpoint/upload failed:"
                    )
                    print(error_message)

                raise RuntimeError(
                    "Rank 0 checkpoint/upload failed."
                )

            # Keep both GPUs synchronized.
            dist.barrier()

        elif not success:
            raise RuntimeError(
                error_message
            )

    # --------------------------------------------------------
    # Production training loop
    # --------------------------------------------------------

    def train(self, max_steps):

        data_iterator = iter(
            self.dataloader
        )

        batch_size = self.dataloader.batch_size

        if batch_size is None:
            batch_size = 1

        model_config = (
            self.model.module.config
            if hasattr(self.model, "module")
            else self.model.config
        )

        context_length = (
            model_config.context_length
        )

        tokens_per_microbatch = (
            batch_size * context_length
        )

        tokens_per_step = (
            tokens_per_microbatch
            * self.gradient_accumulation_steps
            * self.world_size
        )

        if self.is_main_process:

            print()
            print("=" * 60)
            print("GPT-355M KAGGLE TRAINING")
            print("=" * 60)
            print(
                f"Starting from step: "
                f"{self.global_step}"
            )
            print(
                f"Target step: "
                f"{max_steps}"
            )
            print(
                f"Upload interval: "
                f"{self.upload_interval}"
            )
            print(
                f"Tokens/update: "
                f"{tokens_per_step:,}"
            )
            print("=" * 60)
            print()

        remaining_steps = (
            max_steps - self.global_step
        )

        progress = None

        if self.is_main_process:
            progress = tqdm(
                total=remaining_steps,
                desc="Training",
                unit="step",
            )

        while self.global_step < max_steps:

            start_time = time.time()

            total_loss = 0.0

            for micro_step in range(
                self.gradient_accumulation_steps
            ):

                try:
                    batch = next(
                        data_iterator
                    )

                except StopIteration:
                    data_iterator = iter(
                        self.dataloader
                    )
                    batch = next(
                        data_iterator
                    )

                is_last_microbatch = (
                    micro_step
                    == self.gradient_accumulation_steps - 1
                )

                loss = self.train_step(
                    batch,
                    sync_gradients=is_last_microbatch,
                )

                total_loss += loss.item()

            self.optimizer_step()

            self.global_step += 1

            elapsed = (
                time.time() - start_time
            )

            average_loss = total_loss

            if self.scheduler is not None:
                learning_rate = (
                    self.scheduler
                    .get_last_lr()[0]
                )
            else:
                learning_rate = (
                    self.optimizer
                    .param_groups[0]["lr"]
                )

            tokens_per_second = (
                tokens_per_step
                / max(elapsed, 1e-8)
            )

            if self.is_main_process:

                progress.update(1)

                progress.set_postfix(
                    loss=f"{average_loss:.4f}",
                    lr=f"{learning_rate:.2e}",
                )

                if (
                    self.global_step
                    % self.log_interval
                    == 0
                ):

                    print(
                        f"\nStep {self.global_step:6d} | "
                        f"Loss {average_loss:.4f} | "
                        f"LR {learning_rate:.6g} | "
                        f"{tokens_per_second:,.0f} tok/s | "
                        f"{elapsed:.2f}s"
                    )

                    if self.logger is not None:
                        self.logger.log(
                            "train/loss",
                            average_loss,
                            self.global_step,
                        )

                        self.logger.log(
                            "train/learning_rate",
                            learning_rate,
                            self.global_step,
                        )

                        self.logger.log(
                            "train/tokens_per_second",
                            tokens_per_second,
                            self.global_step,
                        )

            # ------------------------------------------------
            # Rolling checkpoint + Kaggle upload
            # ------------------------------------------------

            if (
                self.upload_interval > 0
                and self.global_step
                % self.upload_interval
                == 0
            ):

                self.checkpoint_and_upload(
                    average_loss
                )

        if progress is not None:
            progress.close()

        # ----------------------------------------------------
        # Release HuggingFace streaming iterator
        # ----------------------------------------------------

        del data_iterator

        import gc
        gc.collect()

        # ----------------------------------------------------
        # Final checkpoint
        # ----------------------------------------------------

        if (
            self.global_step
            % self.upload_interval
            != 0
        ):
            self.checkpoint_and_upload(
                average_loss
            )

        if self.is_main_process:
            print()
            print("=" * 60)
            print("Training batch complete.")
            print(
                f"Final step: "
                f"{self.global_step}"
            )
            print("=" * 60)


        
        
        # ----------------------------------------------------
        # Final checkpoint
        # ----------------------------------------------------

        # If the batch ended between upload boundaries,
        # save/upload the final state.
        if (
            self.global_step
            % self.upload_interval
            != 0
        ):

            self.checkpoint_and_upload(
                average_loss
            )

        if self.is_main_process:

            print()
            print("=" * 60)
            print(
                f"Training batch complete."
            )
            print(
                f"Final step: "
                f"{self.global_step}"
            )
            print("=" * 60)

Writing kaggle_trainer.py


In [6]:
%%writefile kaggle_train.py

import os

import torch
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP

from config.model_config import GPTConfig
from config.data_config import DataConfig

from data.dataloader import create_dataloader

from model.gpt import GPT

from training.optimizer import create_optimizer
from training.scheduler import create_scheduler
from training.checkpoint import CheckpointManager

from kaggle_config import (
    DATASET_ID,
    STEPS_PER_BATCH,
    UPLOAD_INTERVAL,
    CHECKPOINT_DIR,
    MAX_LOCAL_CHECKPOINTS,
    BATCH_SIZE,
    GRADIENT_ACCUMULATION_STEPS,
    PRECISION,
    USE_TF32,
    LEARNING_RATE,
    WEIGHT_DECAY,
    BETA1,
    BETA2,
    EPS,
    WARMUP_STEPS,
    SCHEDULER_MAX_STEPS,
    MIN_LR_RATIO,
    MAX_GRAD_NORM,
    DATASET_NAME,
    DATASET_SUBSET,
    STREAMING,
    NUM_WORKERS,
    LOG_INTERVAL,
)

from kaggle_trainer import KaggleTrainer


def main():

    # ========================================================
    # torchrun environment
    # ========================================================

    local_rank = int(
        os.environ["LOCAL_RANK"]
    )

    rank = int(
        os.environ["RANK"]
    )

    world_size = int(
        os.environ["WORLD_SIZE"]
    )

    # ========================================================
    # CUDA / NCCL
    # ========================================================

    torch.cuda.set_device(
        local_rank
    )

    dist.init_process_group(
        backend="nccl",
        init_method="env://",
    )

    device = torch.device(
        f"cuda:{local_rank}"
    )

    # ========================================================
    # Main process information
    # ========================================================

    if rank == 0:

        print()
        print("=" * 60)
        print("355M GPT TRAINING")
        print("=" * 60)
        print(
            f"World size: {world_size}"
        )

        for gpu_index in range(
            world_size
        ):

            print(
                f"GPU {gpu_index}: "
                f"{torch.cuda.get_device_name(gpu_index)}"
            )

        print("=" * 60)

    try:

        # ====================================================
        # Model
        # ====================================================

        model_config = GPTConfig()

        model = GPT(
            model_config
        )

        model.to(device)

        parameter_count = sum(
            parameter.numel()
            for parameter in model.parameters()
        )

        if rank == 0:
            print(
                f"Parameters: "
                f"{parameter_count:,}"
            )

        # ====================================================
        # DDP
        # ====================================================

        model = DDP(
            model,
            device_ids=[local_rank],
            output_device=local_rank,
        )

        # ====================================================
        # Data
        # ====================================================

        data_config = DataConfig(
            dataset_name=DATASET_NAME,
            dataset_subset=DATASET_SUBSET,
            streaming=STREAMING,
            num_workers=NUM_WORKERS,
        )

        dataloader = create_dataloader(
            data_config,
            model_config,
            BATCH_SIZE,
        )

        # ====================================================
        # Optimizer
        # ====================================================

        optimizer = create_optimizer(
            model,
            optimizer_name="adamw",
            learning_rate=LEARNING_RATE,
            weight_decay=WEIGHT_DECAY,
            betas=(BETA1, BETA2),
            eps=EPS,
            use_fused=True,
        )

        # ====================================================
        # Scheduler
        # ====================================================

        scheduler = create_scheduler(
            optimizer,
            warmup_steps=WARMUP_STEPS,
            max_steps=SCHEDULER_MAX_STEPS,
            min_lr_ratio=MIN_LR_RATIO,
        )

        # ====================================================
        # Checkpoint manager
        # ====================================================

        checkpoint_manager = CheckpointManager(
            directory=CHECKPOINT_DIR,
            max_checkpoints=MAX_LOCAL_CHECKPOINTS,
        )

        checkpoint_path = os.path.join(
            CHECKPOINT_DIR,
            "checkpoint_latest.pt",
        )

        if not os.path.exists(
            checkpoint_path
        ):

            raise FileNotFoundError(
                "\nNo checkpoint found!\n\n"
                f"Expected:\n"
                f"{os.path.abspath(checkpoint_path)}\n\n"
                "Download the Kaggle checkpoint "
                "before launching training."
            )

        # ====================================================
        # Trainer
        # ====================================================

        trainer = KaggleTrainer(
            model=model,
            dataloader=dataloader,
            optimizer=optimizer,
            scheduler=scheduler,

            gradient_accumulation_steps=(
                GRADIENT_ACCUMULATION_STEPS
            ),

            max_grad_norm=MAX_GRAD_NORM,

            precision=PRECISION,
            use_tf32=USE_TF32,

            checkpoint_manager=checkpoint_manager,
            checkpoint_interval=0,

            eval_dataloader=None,
            eval_interval=0,

            logger=None,

            local_rank=local_rank,
            rank=rank,
            world_size=world_size,

            log_interval=LOG_INTERVAL,

            upload_dataset_id=DATASET_ID,
            upload_interval=UPLOAD_INTERVAL,
        )

        # ====================================================
        # Load checkpoint on EVERY rank
        # ====================================================

        model_to_load = model.module

        checkpoint = checkpoint_manager.load(
            path=checkpoint_path,
            model=model_to_load,
            optimizer=optimizer,
            scheduler=scheduler,
            scaler=trainer.scaler,
            device=device,
        )

        # CheckpointManager returns:
        #     (step, loss)

        start_step, checkpoint_loss = checkpoint

        trainer.global_step = start_step

        if rank == 0:

            print()
            print("=" * 60)
            print("CHECKPOINT LOADED")
            print("=" * 60)
            print(
                f"Resume step: {start_step}"
            )
            print(
                f"Checkpoint loss: "
                f"{checkpoint_loss:.6f}"
            )
            print(
                f"Next step: "
                f"{start_step + 1}"
            )
            print("=" * 60)

        # Make sure both GPUs loaded before training.
        dist.barrier()

        # ====================================================
        # Determine target step
        # ====================================================

        target_step = (
            start_step
            + STEPS_PER_BATCH
        )

        if rank == 0:

            print()
            print(
                f"Training {STEPS_PER_BATCH} "
                f"additional optimizer steps."
            )
            print(
                f"Target global step: "
                f"{target_step}"
            )
            print()

        # ====================================================
        # TRAIN
        # ====================================================

        trainer.train(
            max_steps=target_step
        )

        # ====================================================
        # Clean up HuggingFace streaming resources
        # ====================================================

        if rank == 0:
            print("\nTraining finished. Cleaning up data pipeline...")

        # Release the iterator that owns the HF streaming
        # network/background resources.
        del trainer.dataloader

        # Give Python a chance to release the underlying
        # streaming dataset/network resources.
        import gc
        gc.collect()

        # Make sure both ranks have finished cleanup.
        dist.barrier()

        if rank == 0:
            print("Data pipeline cleaned up.")
            print("Shutting down distributed training...")

    finally:

        if (
            dist.is_available()
            and dist.is_initialized()
        ):
            dist.destroy_process_group()


if __name__ == "__main__":
    main()

Writing kaggle_train.py


In [7]:
import os

for filename in [
    "kaggle_config.py",
    "kaggle_trainer.py",
    "kaggle_train.py",
]:
    path = os.path.join(
        "/kaggle/working/355m-LLm-Training",
        filename,
    )

    print(
        filename,
        "->",
        os.path.exists(path),
    )

kaggle_config.py -> True
kaggle_trainer.py -> True
kaggle_train.py -> True


In [8]:
!torchrun --standalone --nproc_per_node=2 /kaggle/working/355m-LLm-Training/kaggle_train.py

W0912 07:44:48.829000 49 torch/distributed/run.py:852] 
W0912 07:44:48.829000 49 torch/distributed/run.py:852] *****************************************
W0912 07:44:48.829000 49 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0912 07:44:48.829000 49 torch/distributed/run.py:852] *****************************************
[W912 07:44:49.508328082 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
Traceback (most recent call last):
  File "/kaggle/working/355m-LLm-Training/kaggle_train.py", line 355, in <module>
Traceback (most recent call last):
    main()
  File "/kaggle/working/355m-LLm-Training/kaggle_train.py", line 355, in <module>
  File "/kaggle/working/355m-LLm-Training/kaggle_train.py", line 70, in main
    torch.cuda.set_device(
  File "/usr/local/lib/pyt